# Stage C: inference and evaluation

This notebook runs the end-to-end benchmark-style pipeline:
- GPLPR on LR validation images
- GPLPR on SR-restored validation images
- track-level aggregation
- LR vs SR comparison
- submission-style text export


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/lpsrocr')
DATASET_ROOT = Path('/content/drive/MyDrive/icpr2026_lpr/train')
GPLPR_REPO = Path('/content/gplpr')

STAGE_A_OUTPUT = Path('/content/drive/MyDrive/lpsrocr/outputs/stage_a')
STAGE_B_OUTPUT = Path('/content/drive/MyDrive/lpsrocr/outputs/stage_b')
STAGE_C_OUTPUT = Path('/content/drive/MyDrive/lpsrocr/outputs/stage_c')

STAGE_A_RUN_TAG = '<stage_a_run_tag>'
GPLPR_CHECKPOINT = STAGE_A_OUTPUT / 'checkpoints' / STAGE_A_RUN_TAG / 'best_model.pth'
LR_STAGE_DIR = PROJECT_ROOT / 'external_data' / 'stage_c_lr'
SR_STAGE_DIR = PROJECT_ROOT / 'external_data' / 'stage_c_sr'
LR_OUTPUT_DIR = STAGE_C_OUTPUT / 'lr'
SR_OUTPUT_DIR = STAGE_C_OUTPUT / 'sr'
COMPARISON_DIR = STAGE_C_OUTPUT / 'comparison'
SUBMISSION_DIR = STAGE_C_OUTPUT / 'submissions'
RESTORED_MANIFEST = STAGE_B_OUTPUT / 'restored' / 'restoration_manifest.jsonl'
VAL_SPLIT_DIR = PROJECT_ROOT / 'manifests' / 'splits' / 'scenario_b_dev_seed42_n400_v20'


In [ ]:
%cd /content
!git clone <YOUR_PROJECT_REPO_URL> lpsrocr || true
!git clone https://github.com/valfride/gplpr.git gplpr || true
!git clone https://github.com/valfride/lpsr-lacd.git lpsr-lacd || true
%cd /content/lpsrocr
!pip install -r requirements.txt


In [ ]:
!python scripts/stage_c_run_gplpr_on_lr.py \
  --project-root /content/lpsrocr \
  --dataset-root /content/drive/MyDrive/icpr2026_lpr/train \
  --gplpr-repo /content/gplpr \
  --split-dir /content/lpsrocr/manifests/splits/scenario_b_dev_seed42_n400_v20 \
  --gplpr-checkpoint /content/drive/MyDrive/lpsrocr/outputs/stage_a/checkpoints/<stage_a_run_tag>/best_model.pth \
  --stage-dir /content/lpsrocr/external_data/stage_c_lr \
  --output-dir /content/drive/MyDrive/lpsrocr/outputs/stage_c/lr \
  --mode symlink

!python scripts/stage_c_run_gplpr_on_sr.py \
  --project-root /content/lpsrocr \
  --gplpr-repo /content/gplpr \
  --restored-manifest /content/drive/MyDrive/lpsrocr/outputs/stage_b/restored/restoration_manifest.jsonl \
  --gplpr-checkpoint /content/drive/MyDrive/lpsrocr/outputs/stage_a/checkpoints/<stage_a_run_tag>/best_model.pth \
  --stage-dir /content/lpsrocr/external_data/stage_c_sr \
  --output-dir /content/drive/MyDrive/lpsrocr/outputs/stage_c/sr \
  --mode symlink


In [ ]:
!python scripts/stage_c_compare_lr_sr.py \
  --lr-output-dir /content/drive/MyDrive/lpsrocr/outputs/stage_c/lr \
  --sr-output-dir /content/drive/MyDrive/lpsrocr/outputs/stage_c/sr \
  --output-dir /content/drive/MyDrive/lpsrocr/outputs/stage_c/comparison

!python scripts/stage_c_write_submission_like_txt.py \
  --per-track-csv /content/drive/MyDrive/lpsrocr/outputs/stage_c/sr/per_track.csv \
  --output-file /content/drive/MyDrive/lpsrocr/outputs/stage_c/submissions/sr_majority.txt \
  --aggregation-mode majority

print('Stage C outputs are under', STAGE_C_OUTPUT)
